In [6]:
# Test data repair scripts for Chicago

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))
from phi_data_repair import fill_phi_dates

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_top50_sample.parquet")


In [7]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == "San Antonio"]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FILLED'] = False

sub_df_filled = sub_df.copy()
#sub_df_filled = fill_phi_dates(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [8]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 99.2% -> 99.2%
PERMIT_DATE available (all): 15.0% -> 15.0%
FINAL_DATE available (all): 1.6% -> 1.6%
PERMIT_DATE available (active/final): 18.7% -> 18.7%
FINAL_DATE available (final): 0.3% -> 0.3%


In [9]:
#mask = sub_df_filled["FINAL_DATE"].isna()
mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FILLED']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FILLED']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FILLED']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Final
RECORD_TYPE_ORIGINAL: Fire Life Safety System Application
FILE_DATE: 2024-11-27       *Filled: False*
PERMIT_DATE: None   *Filled: False*
FINAL_DATE: None     *Filled: False*
DATES_DATA: 
{
  "date": "2024-11-27",
  "tasks": [
    {
      "events": [
        {
          " on ": "11/27/2024",
          "Due on ": "11/27/2024"
        }
      ]
    },
    {
      "events": [
        {
          " on ": "11/27/2024",
          "Due on ": "11/27/2024"
        }
      ]
    }
  ],
  "status": "Closed",
  "search_data": {
    "Date": "11/27/2024",
    "Action": "",
    "Status": "Closed",
    "Expiration Date": ""
  },
  "more_details": {
    "Application Information": {
      "INSPECTION DETAILS": {
        "Date of Inspection": "11/05/2024",
        "Date Next Inspection is Due": "02/28/2025"
      }
    }
  }
}


In [11]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "date": "2024-11-27",
  "tasks": [
    {
      "name": "Application Intake",
      "events": [
        {
          " by ": "COSA Admin",
          " on ": "11/27/2024",
          "html": "<td>Due on <span>11/27/2024</span>, assigned to <span>TBD</span><br/>Marked as <span>Received Online</span> on <span>11/27/2024</span> by <span>COSA Admin</span></td>",
          "Due on ": "11/27/2024",
          "Marked as ": "Received Online",
          ", assigned to ": "TBD"
        }
      ]
    },
    {
      "name": "Closure",
      "events": [
        {
          " by ": "COSA Admin",
          " on ": "11/27/2024",
          "html": "<td>Due on <span>11/27/2024</span>, assigned to <span>TBD</span><br/>Marked as <span>Closed</span> on <span>11/27/2024</span> by <span>COSA Admin</span></td>",
          "Due on ": "11/27/2024",
          "Marked as ": "Closed",
          ", assigned to ": "TBD"
        }
      ]
    }
  ],
  "status": "Closed",
  "address": "7039 FAIRGROUNDS",
  "deta